# 05. 결과가 우연인가 — 차체 단위 부트스트랩과 같은 위치 안 순열검정
- 검사 정책(점수 상위 ceil(b·n))은 고정하고, 각 기록이 뽑힐 확률을 5시드 평균한 뒤 **차체를 복원추출**해 포착률의 불확실성을 본다.
  같은 부트스트랩 표본에서 두 모델의 차이를 계산한다(대응 비교).
- 같은 위치 안 판별력: 같은 fold·위치 그룹 안에서 점수를 섞는 순열검정. 위치 간 차이를 빼고도 신호가 남는지 본다.
- 시간순 시험(04): 시험 차체 9대를 복원추출.

In [1]:
import numpy as np, pandas as pd
import br
from br import features as F, budget as B
from wl.metrics import inclusion_prob, within_loc_auroc
pd.set_option("display.width", 220)
df = F.load(); y = df.y.values; loc = df["loc"].values; body = df.body.values
oof = pd.read_pickle(br.RES / "polito_oof.pkl").sort_values(["label", "seed", "row"])
NB = 5000; rng = np.random.default_rng(0)
ub = np.unique(body)
draws = rng.integers(0, len(ub), (NB, len(ub)))
W = np.stack([np.bincount(d_, minlength=len(ub)) for d_ in draws])[:, np.searchsorted(ub, body)]  # (NB, n) 행 가중치

def incl(lab, b):
    """5시드 평균 선택 확률."""
    ps = []
    for s, g in oof[oof.label == lab].groupby("seed"):
        ug, k = B.pick_counts(g.fold.values, b)
        ps.append(inclusion_prob(g.score.values, k.astype(float), np.searchsorted(ug, g.fold.values)))
    return np.mean(ps, 0)

def boot_recall(p):
    return (W @ (p * y)) / (W @ y)

def ci(a): return f"{np.mean(a):.3f} [{np.quantile(a, .025):.3f}, {np.quantile(a, .975):.3f}]"

## 1. 10%·20% 검사 포착률의 95% 구간, 그리고 모델 간 차이

In [2]:
LABS = ["위치별 과거 불량률", "CatBoost·기본16", "TabICLv2·기본16", "CatBoost·도메인90", "TabICLv2·도메인90"]
PAIRS = [("CatBoost·도메인90", "위치별 과거 불량률"), ("TabICLv2·도메인90", "위치별 과거 불량률"),
         ("CatBoost·도메인90", "CatBoost·기본16"), ("TabICLv2·도메인90", "TabICLv2·기본16"),
         ("TabICLv2·도메인90", "CatBoost·도메인90")]
out1, out2 = [], []
for b in (0.10, 0.20):
    R = {lab: boot_recall(incl(lab, b)) for lab in LABS}
    for lab in LABS:
        out1.append({"예산": f"{int(b*100)}%", "모델": lab, "포착률 [95% 구간]": ci(R[lab])})
    for a, c in PAIRS:
        d_ = R[a] - R[c]
        out2.append({"예산": f"{int(b*100)}%", "비교": f"{a} − {c}", "차이 [95% 구간]": ci(d_), "차이≤0 비율": f"{(d_ <= 0).mean():.3f}"})
display(pd.DataFrame(out1).pivot(index="모델", columns="예산", values="포착률 [95% 구간]").loc[LABS])
t2 = pd.DataFrame(out2); t2.to_csv(br.RES / "significance_pairs.csv", index=False, encoding="utf-8-sig"); t2

예산,10%,20%
모델,,
위치별 과거 불량률,"0.523 [0.463, 0.596]","0.732 [0.658, 0.812]"
CatBoost·기본16,"0.546 [0.462, 0.629]","0.706 [0.629, 0.775]"
TabICLv2·기본16,"0.618 [0.532, 0.706]","0.839 [0.761, 0.906]"
CatBoost·도메인90,"0.613 [0.539, 0.682]","0.765 [0.716, 0.813]"
TabICLv2·도메인90,"0.682 [0.616, 0.748]","0.841 [0.794, 0.883]"


,예산,비교,차이 [95% 구간],차이≤0 비율
0,10%,CatBoost·도메인90 − 위치별 과거 불량률,"0.090 [0.020, 0.156]",0.007
1,10%,TabICLv2·도메인90 − 위치별 과거 불량률,"0.159 [0.086, 0.230]",0.000
2,10%,CatBoost·도메인90 − CatBoost·기본16,"0.068 [0.000, 0.137]",0.027
3,10%,TabICLv2·도메인90 − TabICLv2·기본16,"0.064 [-0.030, 0.144]",0.089
4,10%,TabICLv2·도메인90 − CatBoost·도메인90,"0.069 [0.013, 0.135]",0.009
5,20%,CatBoost·도메인90 − 위치별 과거 불량률,"0.032 [-0.046, 0.105]",0.190
6,20%,TabICLv2·도메인90 − 위치별 과거 불량률,"0.108 [0.046, 0.165]",0.001
7,20%,CatBoost·도메인90 − CatBoost·기본16,"0.059 [-0.007, 0.129]",0.045
8,20%,TabICLv2·도메인90 − TabICLv2·기본16,"0.002 [-0.048, 0.057]",0.507
9,20%,TabICLv2·도메인90 − CatBoost·도메인90,"0.076 [0.037, 0.116]",0.000


## 2. 위치 73·84를 뺀 나머지 불량(54개)에서도 위치 기준보다 나은가 (10%)

In [3]:
hot = np.isin(loc, [73, 84])
def incl_sub(lab, b, m):
    ps = []
    for s, g in oof[oof.label == lab].groupby("seed"):
        f = g.fold.values[m]; ug, k = B.pick_counts(f, b)
        ps.append(inclusion_prob(g.score.values[m], k.astype(float), np.searchsorted(ug, f)))
    return np.mean(ps, 0)
Wm, ym = W[:, ~hot], y[~hot]
R = {lab: (Wm @ (incl_sub(lab, .1, ~hot) * ym)) / (Wm @ ym) for lab in ["위치별 과거 불량률", "CatBoost·도메인90", "TabICLv2·도메인90"]}
for lab, r in R.items(): print(f"{lab:20s} {ci(r)}")
for a in ["CatBoost·도메인90", "TabICLv2·도메인90"]:
    d_ = R[a] - R["위치별 과거 불량률"]; print(f"{a} − 위치 기준: {ci(d_)}, 차이≤0 비율 {(d_<=0).mean():.3f}")

위치별 과거 불량률           0.352 [0.253, 0.436]
CatBoost·도메인90       0.481 [0.362, 0.578]
TabICLv2·도메인90       0.574 [0.453, 0.667]
CatBoost·도메인90 − 위치 기준: 0.129 [0.041, 0.217], 차이≤0 비율 0.003
TabICLv2·도메인90 − 위치 기준: 0.222 [0.150, 0.295], 차이≤0 비율 0.000


## 3. 같은 fold·위치 안 판별력 — 순열검정 (시드별)
정상·불량이 함께 있는 fold×위치 그룹에서만 계산된다. 귀무가설: 그룹 안 점수 순서가 무작위.

In [4]:
NP_ = 2000
rows = []
for lab in ["위치별 과거 불량률", "CatBoost·기본16", "CatBoost·도메인90", "TabICLv2·도메인90"]:
    for s, g in oof[oof.label == lab].groupby("seed"):
        grp = np.char.add(g.fold.values.astype(str), "_" + loc.astype(str))
        sc = g.score.values
        obs = within_loc_auroc(y, sc, grp)
        gi = pd.Series(grp).groupby(grp).indices
        mixed = [ix for ix in gi.values() if 0 < y[ix].sum() < len(ix)]
        npair = sum(int(y[ix].sum() * (len(ix) - y[ix].sum())) for ix in mixed)
        null = []
        for _ in range(NP_):
            sp = sc.copy()
            for ix in mixed: sp[ix] = sc[rng.permutation(ix)]
            null.append(within_loc_auroc(y, sp, grp))
        null = np.array(null)
        rows.append(dict(모델=lab, seed=s, AUROC=obs, p값=(np.sum(null >= obs) + 1) / (NP_ + 1),
                         불량_포함그룹=len(mixed), 비교쌍=npair))
perm = pd.DataFrame(rows); perm.to_csv(br.RES / "significance_withinloc.csv", index=False, encoding="utf-8-sig")
perm.groupby("모델")[["AUROC", "p값", "불량_포함그룹", "비교쌍"]].agg(["mean", "min", "max"]).round(3)

AUROC                   p값               불량_포함그룹           비교쌍         
                 mean    min    max   mean    min    max    mean min max  mean min  max
모델                                                                                     
CatBoost·기본16   0.608  0.553  0.674  0.088  0.004  0.209    49.4  44  53  91.8  70  102
CatBoost·도메인90  0.646  0.596  0.717  0.024  0.000  0.078    49.4  44  53  91.8  70  102
TabICLv2·도메인90  0.732  0.647  0.814  0.003  0.000  0.012    49.4  44  53  91.8  70  102
위치별 과거 불량률      0.500  0.500  0.500  1.000  1.000  1.000    49.4  44  53  91.8  70  102

## 4. 시간순 시험(2024년 5일, 불량 51개) — 무작위보다 나은가 (10%)
시험 차체 9대를 복원추출. 무작위 기대 포착률은 날짜별 ceil 때문에 약 11%.

In [5]:
pr = pd.read_pickle(br.RES / "daily_aggregate_preds.pkl")
te = np.sort(pr.row.unique()); yte = y[te]; dte = df.date.dt.strftime("%Y-%m-%d").values[te]; bte = body[te]
ubt = np.unique(bte); dr = rng.integers(0, len(ubt), (NB, len(ubt)))
Wt = np.stack([np.bincount(x, minlength=len(ubt)) for x in dr])[:, np.searchsorted(ubt, bte)]
ug, k = B.pick_counts(dte, .10); gi = np.searchsorted(ug, dte)
p_rand = (k / np.bincount(gi))[gi]
r_rand = (Wt @ (p_rand * yte)) / (Wt @ yte)
rows = []
for (proto, lab), g in pr.groupby(["proto", "label"]):
    if lab == "일정 불량률": continue
    ps = [inclusion_prob(gg.set_index("row").loc[te].p.values, k.astype(float), gi) for _, gg in g.groupby("seed")]
    r = (Wt @ (np.mean(ps, 0) * yte)) / (Wt @ yte)
    d_ = r - r_rand
    rows.append({"모델": f"{lab} [{proto}]", "포착률 [95%]": ci(r), "무작위 대비 차이 [95%]": ci(d_), "차이≤0 비율": f"{(d_<=0).mean():.3f}"})
print("무작위 기대 포착률", ci(r_rand))
pd.DataFrame(rows)

무작위 기대 포착률 0.111 [0.101, 0.135]


,모델,포착률 [95%],무작위 대비 차이 [95%],차이≤0 비율
0,PVC MLP·기본16 [A],"0.162 [0.081, 0.231]","0.051 [-0.030, 0.120]",0.100
1,PVC MLP·도메인90 [A],"0.197 [0.124, 0.269]","0.086 [0.009, 0.160]",0.016
2,비율맞춤 MLP·도메인90 [A],"0.277 [0.189, 0.373]","0.166 [0.071, 0.266]",0.000
3,집계 로지스틱·도메인90 [A],"0.250 [0.146, 0.338]","0.138 [0.024, 0.232]",0.010
4,PVC MLP·기본16 [B],"0.193 [0.128, 0.243]","0.082 [0.012, 0.137]",0.012
5,PVC MLP·도메인90 [B],"0.267 [0.183, 0.342]","0.156 [0.064, 0.237]",0.001
6,비율맞춤 MLP·도메인90 [B],"0.282 [0.166, 0.375]","0.170 [0.048, 0.269]",0.005
7,집계 로지스틱·도메인90 [B],"0.212 [0.106, 0.314]","0.101 [-0.007, 0.207]",0.037
8,CatBoost·도메인90 (개별 정답) [참고],"0.272 [0.191, 0.355]","0.161 [0.073, 0.248]",0.000
9,TabICLv2·도메인90 (개별 정답) [참고],"0.409 [0.312, 0.507]","0.298 [0.189, 0.403]",0.000
